# NGC 1068 FOV-cut significance comparison

This notebook applies each field-of-view cut consistently to the spacecraft orientation and the unbinned DC4 background. It then injects NGC 1068 using the cut orientation, bins the cut background in memory, and compares source and background counts. No intermediate products are saved.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.coordinates import SkyCoord
from astromodels import Cutoff_powerlaw
from threeML import Model, PointSource

from cosipy import BinnedData, SourceInjector, SpacecraftHistory
from cosipy.event_selection import GoodTimeInterval

%matplotlib inline

23:11:16 WARNING   The naima package is not available. Models that depend on it will not be         ]8;id=754260;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=52679;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#43\43]8;;\
                  available                                                                                        

         WARNING   The GSL library or the pygsl wrapper cannot be loaded. Models that depend on it  ]8;id=453796;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py\functions.py]8;;\:]8;id=288759;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/functions.py#65\65]8;;\
                  will not be available.                                                                           

         WARNING   The ebltable package is not available. Models that depend on it will not be     ]8;id=600204;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py\absorption.py]8;;\:]8;id=302204;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/astromodels/functions/functions_1D/absorption.py#33\33]8;;\
                  available                                                                                        

23:11:17 INFO      Starting 3ML!                                                                     ]8;id=517288;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=435502;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#44\44]8;;\

         WARNING   WARNINGs here are NOT errors                                                      ]8;id=111112;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=908531;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#45\45]8;;\

...

         WARNING   Could not import plugin FermiLATLike.py. Do you have the relative instrument     ]8;id=405116;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=145606;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#126\126]8;;\
                  software installed and configured?                                                               

         WARNING   No fermitools installed                                              ]8;id=851243;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py\lat_transient_builder.py]8;;\:]8;id=410342;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/utils/data_builders/fermi/lat_transient_builder.py#44\44]8;;\

         WARNING   Env. variable OMP_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=311675;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=423135;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable MKL_NUM_THREADS is not set. Please set it to 1 for optimal         ]8;id=575986;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=748374;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

         WARNING   Env. variable NUMEXPR_NUM_THREADS is not set. Please set it to 1 for optimal     ]8;id=613319;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py\__init__.py]8;;\:]8;id=17100;file:///Users/parshadkp/.pyenv/versions/cosipy312/lib/python3.12/site-packages/threeML/__init__.py#335\335]8;;\
                  performances in 3ML                                                                              

## Input files

In [ ]:
response_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/ResponseContinuum.o3.e100_10000.b10log.s10396905069491.m2284.filtered.nonsparse.binnedimaging.imagingresponse.h5"
)
orientation_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/DC4_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.fits"
)
background_path = Path(
    "/Users/parshadkp/Library/CloudStorage/OneDrive-ClemsonUniversity/COSI/Software_Files/DC4_Files/Background/Total_DC4_BG_3months_unbinned_data_filtered_with_SAAcut_withSAAbck.fits"
)
binning_config = Path(
    "/Users/parshadkp/Software/cosipy/docs/tutorials/source_injector/inputs.yaml"
)

for path in (response_path, orientation_path, background_path, binning_config):
    if not path.exists():
        raise FileNotFoundError(path)

## NGC 1068 source model

In [ ]:
source_name = "NGC1068"
source_coord = SkyCoord(l=172.104 * u.deg, b=-51.934 * u.deg, frame="galactic")

spectrum = Cutoff_powerlaw()
spectrum.K.value = 3.1e-1
spectrum.K.unit = 1 / (u.cm**2 * u.s * u.keV)
spectrum.piv.value = 1.0
spectrum.piv.unit = u.keV
spectrum.xc.value = 200.0
spectrum.xc.unit = u.keV
spectrum.index.value = -1.92

point_source = PointSource(
    source_name,
    l=source_coord.l.deg,
    b=source_coord.b.deg,
    spectral_shape=spectrum,
)
model = Model(point_source)

## Load the full orientation and unbinned background once

The scan below constructs a new GTI for every angle. The same GTI is applied to both inputs before source injection or background binning.

In [ ]:
orientation = SpacecraftHistory.open(orientation_path)

background_reader = BinnedData(str(binning_config))
background_events = background_reader.get_dict(str(background_path))

injector = SourceInjector(response_path=response_path)
total_livetime = orientation.cumulative_livetime().to_value(u.s)

print(f"Orientation samples: {len(orientation.obstime):,}")
print(f"Unbinned background events: {len(background_events['TimeTags']):,}")
print(f"Total livetime: {total_livetime:,.1f} s")

## Scan the FOV cuts

The primary metric is the Asimov counting significance

$$Z_A = \sqrt{2\left[(S+B)\ln(1+S/B)-S\right]}.$$

For reference, the simpler $S/\sqrt{B}$ and $S/\sqrt{S+B}$ metrics are also reported. This is a counts-only optimization; a full likelihood analysis may prefer a somewhat different cut.

In [ ]:
fov_angles = np.arange(20, 91, 5) * u.deg
earth_occ = True


def histogram_counts(histogram):
    contents = histogram.contents
    if hasattr(contents, "value"):
        contents = contents.value
    return float(np.asarray(contents).sum())


def asimov_significance(source_counts, background_counts):
    if source_counts <= 0:
        return 0.0
    if background_counts <= 0:
        return np.inf
    return np.sqrt(
        2.0
        * (
            (source_counts + background_counts)
            * np.log1p(source_counts / background_counts)
            - source_counts
        )
    )


rows = []
energy_spectra = {}

for max_offaxis in fov_angles:
    source_gti = GoodTimeInterval.from_pointing_cut(
        source_coord,
        orientation,
        max_offaxis,
        earth_occ=earth_occ,
    )

    cut_orientation = orientation.apply_gti(source_gti)
    cut_livetime = cut_orientation.cumulative_livetime().to_value(u.s)

    background_mask = source_gti.contains(background_events["TimeTags"])
    n_background_events = int(np.count_nonzero(background_mask))
    if n_background_events == 0 or cut_livetime <= 0:
        print(f"Skipping {max_offaxis.value:.0f} deg: no selected exposure/background")
        continue

    cut_background_events = {
        key: values[background_mask]
        for key, values in background_events.items()
    }

    background_binner = BinnedData(str(binning_config))
    background_binner.cosi_dataset = cut_background_events
    background_binner.tmin = float(np.min(cut_background_events["TimeTags"]))
    background_binner.tmax = float(np.max(cut_background_events["TimeTags"]))
    background_binner.get_binned_data(
        make_binning_plots=False,
        show_plots=False,
    )
    background_hist = background_binner.binned_data

    source_hist = injector.inject_model(
        model=model,
        orientation=cut_orientation,
        make_spectrum_plot=False,
        earth_occ=earth_occ,
    )

    source_counts = histogram_counts(source_hist)
    background_counts = histogram_counts(background_hist)
    significance = asimov_significance(source_counts, background_counts)

    angle = float(max_offaxis.to_value(u.deg))
    rows.append(
        {
            "fov_cut_deg": angle,
            "gti_intervals": len(source_gti),
            "livetime_s": cut_livetime,
            "livetime_fraction": cut_livetime / total_livetime,
            "selected_unbinned_background_events": n_background_events,
            "source_counts": source_counts,
            "background_counts": background_counts,
            "asimov_sigma": significance,
            "s_over_sqrt_b": source_counts / np.sqrt(background_counts),
            "s_over_sqrt_s_plus_b": source_counts / np.sqrt(source_counts + background_counts),
        }
    )
    energy_spectra[angle] = (
        source_hist.project("Em"),
        background_hist.project("Em"),
    )

    print(
        f"{angle:4.0f} deg: S={source_counts:,.1f}, B={background_counts:,.1f}, "
        f"Z_A={significance:.3f} sigma"
    )

results = pd.DataFrame(rows).sort_values("fov_cut_deg").reset_index(drop=True)
results

## Identify and visualize the best cut

In [ ]:
if results.empty:
    raise RuntimeError("No FOV cut produced usable source and background histograms.")

best_index = results["asimov_sigma"].idxmax()
best = results.loc[best_index]
best_angle = float(best["fov_cut_deg"])

print(
    f"Best counts-only FOV cut: {best_angle:.0f} deg\n"
    f"Source counts: {best['source_counts']:,.1f}\n"
    f"Background counts: {best['background_counts']:,.1f}\n"
    f"Asimov significance: {best['asimov_sigma']:.3f} sigma\n"
    f"Livetime fraction: {best['livetime_fraction']:.3f}"
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

axes[0].plot(results["fov_cut_deg"], results["asimov_sigma"], "o-", lw=2)
axes[0].axvline(best_angle, color="0.3", ls="--", label=f"Best: {best_angle:.0f}°")
axes[0].set_xlabel("Maximum off-axis angle [deg]")
axes[0].set_ylabel("Asimov counting significance [$\\sigma$]")
axes[0].grid(alpha=0.3)
axes[0].legend(frameon=False)

source_energy, background_energy = energy_spectra[best_angle]
energy_edges = source_energy.axes["Em"].bounds
axes[1].stairs(source_energy.contents, energy_edges, lw=2, label="Injected NGC 1068")
axes[1].stairs(background_energy.contents, energy_edges, lw=2, label="Cut background")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Measured energy [keV]")
axes[1].set_ylabel("Counts")
axes[1].set_title(f"Best {best_angle:.0f}° FOV cut")
axes[1].grid(alpha=0.3)
axes[1].legend(frameon=False)

The optimal angle above is based only on total source and background counts. For the final science analysis, verify the choice with the same likelihood, energy range, nuisance parameters, and response treatment used for the reported source significance.